# TUM-VIE: New Fusion Models Ablation

## Research Question
Which fusion architecture best regresses 3-DoF **translation** from event cameras + IMU
on the TUM-VIE `mocap-6dof` recording, using an **in-distribution** train/val/test split?

## Why This Matters
Spyx now ships three qualitatively different fusion strategies for visual–inertial odometry.
This notebook provides the canonical ablation to understand their relative strengths:

| Architecture | Fusion Style | Key Inductive Bias |
|---|---|---|
| **IMUConditionedVisualSNN** | Late gating | IMU shifts the visual readout at inference |
| **VisualIMURecurrentFusionBlock** | Recurrent (RLIF) | Temporal integration of visual + IMU latents |
| **KalmanStyleSpikingFusionSurrogate** | Predict-correct | Differentiable Kalman-like innovation loop |

## Experimental Setup
- Single recording (`mocap-6dof`), 80/10/10 train/val/test split
- **Target**: 3-DoF translation xyz (metres), z-scored for training stability
- Per-epoch val tracking; test evaluation at convergence
- Spike-rate time-series per model (energy efficiency proxy)
- Model complexity: param count × latency tradeoff

---
*Results are appended to `experiments/quant_pipeline/pipeline_runs.jsonl`.*

In [ ]:
from __future__ import annotations
from pathlib import Path
import time, json, datetime
from collections import defaultdict

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import optax
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from tonic import transforms
from tonic.datasets import TUMVIE

import spyx.models as fm

plt.rcParams.update({"figure.dpi": 120, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

# ── Experiment hyper-parameters ───────────────────────────────────────────────
SEED           = 11
RECORDING      = "mocap-6dof"
SPATIAL_FACTOR = 0.2
N_TIME_BINS    = 20
N_SAMPLES      = 150
BATCH_SIZE     = 8
EPOCHS         = 25
LR             = 2e-3
LATENT_DIM     = 64    # shared visual latent size across fusion models

# ── I/O ───────────────────────────────────────────────────────────────────────
data_root    = Path("../data").resolve()
FIG_DIR      = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)
RESULTS_FILE = Path("../../experiments/quant_pipeline/pipeline_runs.jsonl").resolve()

MODEL_COLORS = {
    "IMUConditioned":    "#66c2a5",
    "RecurrentFusion":   "#fc8d62",
    "KalmanFusion":      "#8da0cb",
}

# ── Tonic dataset + transform ─────────────────────────────────────────────────
sensor      = TUMVIE.sensor_size
sensor_small = (int(sensor[0] * SPATIAL_FACTOR),
                int(sensor[1] * SPATIAL_FACTOR),
                sensor[2])
tfm = transforms.Compose([
    transforms.Downsample(spatial_factor=SPATIAL_FACTOR),
    transforms.ToFrame(sensor_size=sensor_small, n_time_bins=N_TIME_BINS),
])
ds = TUMVIE(save_to=str(data_root), recording=RECORDING, transform=tfm)
print(f"Dataset size: {len(ds):,}  |  sensor_small: {sensor_small}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Data Loading
# ─────────────────────────────────────────────────────────────────────────────
def build_arrays(limit: int = N_SAMPLES):
    xs, us, ys = [], [], []
    n = min(limit, len(ds))
    for i in range(n):
        d, t = ds[i]
        left  = np.asarray(d["events_left"],  dtype=np.float32)   # (T, H, W, P)
        right = np.asarray(d["events_right"], dtype=np.float32)
        x     = np.concatenate([left, right], axis=1)              # (T, 2H, W, P)

        imu = np.asarray(d["imu"], dtype=np.float32)
        imu = imu.mean(axis=0) if (imu.ndim == 2 and imu.shape[0] > 0) \
              else np.zeros(6, dtype=np.float32)

        mocap = np.asarray(t["mocap"], dtype=np.float32)
        mocap = mocap[-1] if mocap.ndim == 2 else mocap
        y = mocap[:3]   # translation xyz only

        xs.append(x); us.append(imu); ys.append(y)

    return np.stack(xs), np.stack(us), np.stack(ys)

print("Building arrays …")
X_all, U_all, Y_all = build_arrays()

# ── Infer model input dims ────────────────────────────────────────────────────
N_TOTAL, T_BINS, H_SPATIAL, W_SPATIAL, N_CHANNELS = X_all.shape
INPUT_HW       = (H_SPATIAL, W_SPATIAL)
INPUT_CHANNELS = N_CHANNELS
IMU_DIM        = U_all.shape[-1]   # 6
OUTPUT_DIM     = Y_all.shape[-1]   # 3

print(f"X: {X_all.shape}  U: {U_all.shape}  Y: {Y_all.shape}")
print(f"input_hw={INPUT_HW}, channels={INPUT_CHANNELS}, T={T_BINS}, imu_dim={IMU_DIM}")

# ── 80/10/10 train/val/test split ─────────────────────────────────────────────
idx     = np.arange(N_TOTAL)
idx_tr, idx_te = train_test_split(idx, test_size=0.2, random_state=SEED)
idx_tr, idx_va = train_test_split(idx_tr, test_size=0.125, random_state=SEED + 1)
# → 70% train, ~10% val, ~20% test (if 150 samples: ~105 / 15 / 30)

def _split(a, i): return jnp.asarray(a[i])

Xtr_j, Utr_j, Ytr_j = _split(X_all, idx_tr), _split(U_all, idx_tr), jnp.asarray(Y_all[idx_tr])
Xva_j, Uva_j, Yva_j = _split(X_all, idx_va), _split(U_all, idx_va), jnp.asarray(Y_all[idx_va])
Xte_j, Ute_j, Yte_j = _split(X_all, idx_te), _split(U_all, idx_te), jnp.asarray(Y_all[idx_te])

print(f"train: {Xtr_j.shape[0]} | val: {Xva_j.shape[0]} | test: {Xte_j.shape[0]}")

# ── Target normalisation (z-score using train statistics) ─────────────────────
Y_mean = float(np.mean(Y_all[idx_tr]))
Y_std  = float(np.std(Y_all[idx_tr]))  + 1e-6
Ytr_z  = jnp.asarray((Y_all[idx_tr]  - Y_mean) / Y_std)
Yva_z  = jnp.asarray((Y_all[idx_va]  - Y_mean) / Y_std)
Yte_z  = jnp.asarray((Y_all[idx_te]  - Y_mean) / Y_std)

# ── Batch helpers (time-major) ────────────────────────────────────────────────
def _time_major(X, idx): return jnp.swapaxes(X[idx], 0, 1)   # (T, B, H, W, C)

def make_batch_ab(X, U, Y, bs, key):
    idx_b = jax.random.choice(key, X.shape[0], shape=(bs,), replace=False)
    return _time_major(X, idx_b), U[idx_b], Y[idx_b]

def iter_batches(X, U, Y):
    n = (X.shape[0] // BATCH_SIZE) * BATCH_SIZE
    for b in range(0, n, BATCH_SIZE):
        yield _time_major(X, np.arange(b, b + BATCH_SIZE)), \
              U[b:b+BATCH_SIZE], Y[b:b+BATCH_SIZE]

# ── Visualise target distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ci, (ax, label) in enumerate(zip(axes, ["tx (m)", "ty (m)", "tz (m)"])):
    ax.hist(Y_all[idx_tr, ci], bins=20, alpha=0.7, color="#377eb8", label="train", density=True)
    ax.hist(Y_all[idx_va, ci], bins=20, alpha=0.7, color="#4daf4a", label="val",   density=True)
    ax.hist(Y_all[idx_te, ci], bins=20, alpha=0.7, color="#e41a1c", label="test",  density=True)
    ax.set_title(label); ax.set_xlabel("value (m)")
    if ci == 0: ax.legend(fontsize=7)
fig.suptitle(f"Translation Target Distribution — {RECORDING}", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "ablation_target_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. Model Definitions  (all use correct ConvConfig / fusion config APIs)
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTANT — forward function input conventions:
#   x_seq   : (T, batch, H, W, C)          time-major event frames
#   imu     : (batch, 6)                    mean-pooled IMU per sample
#   imu_seq : (T, batch, 6)                 IMU repeated along time axis
#   traj_seq: (T, batch, 3)                 prior trajectory (zeros = no prior)

def imu_conditioned_fwd(x_seq, imu):
    """IMUConditionedVisualSNN: late gating of visual SNN by IMU MLP branch."""
    vis_cfg = fm.ConvConfig(
        input_hw=INPUT_HW, input_channels=INPUT_CHANNELS,
        channels1=32, channels2=64, output_dim=OUTPUT_DIM,
    )
    cfg = fm.IMUConditionedConfig(
        vision_cfg=vis_cfg, imu_dim=IMU_DIM, imu_hidden=128)
    return fm.IMUConditionedVisualSNN(cfg)(x_seq, imu)   # → (B,3), aux


def recurrent_fusion_fwd(x_seq, imu_seq, traj_seq):
    """VisualIMURecurrentFusion: RLIF core integrates visual+IMU+traj latents."""
    vis_cfg = fm.ConvConfig(
        input_hw=INPUT_HW, input_channels=INPUT_CHANNELS,
        channels1=24, channels2=48, output_dim=LATENT_DIM,
    )
    cfg = fm.VisualIMURecurrentConfig(
        vision_cfg=vis_cfg, imu_dim=IMU_DIM,
        traj_dim=OUTPUT_DIM, hidden_dim=128, output_dim=OUTPUT_DIM,
    )
    return fm.VisualIMURecurrentFusionBlock(cfg)(x_seq, imu_seq, traj_seq)


def kalman_fusion_fwd(x_seq, imu_seq):
    """KalmanStyleFusion: ConvLIF extracts visual latents, then predict+correct."""
    # Step A: extract per-timestep visual latents via shared ConvLIF backbone
    vis_cfg = fm.ConvConfig(
        input_hw=INPUT_HW, input_channels=INPUT_CHANNELS,
        channels1=24, channels2=48, output_dim=LATENT_DIM,
    )
    _, vis_aux = fm.ConvLIFSNN(vis_cfg)(x_seq)
    vis_seq = vis_aux["logits_seq"]   # (T, batch, LATENT_DIM)

    # Step B: Kalman predict→correct on imu_seq vs vis_seq
    cfg = fm.KalmanFusionConfig(latent_dim=LATENT_DIM, output_dim=OUTPUT_DIM)
    return fm.KalmanStyleSpikingFusionSurrogate(cfg)(vis_seq, imu_seq)


# ── Build haiku-transformed models per architecture ────────────────────────────
def _make_transformed(name: str):
    if name == "IMUConditioned":
        return hk.without_apply_rng(hk.transform(
            lambda x, imu: imu_conditioned_fwd(x, imu))), "imu"
    if name == "RecurrentFusion":
        return hk.without_apply_rng(hk.transform(
            lambda x, imu_seq, traj_seq: recurrent_fusion_fwd(x, imu_seq, traj_seq))), "recurrent"
    if name == "KalmanFusion":
        return hk.without_apply_rng(hk.transform(
            lambda x, imu_seq: kalman_fusion_fwd(x, imu_seq))), "kalman"
    raise ValueError(name)


# ─────────────────────────────────────────────────────────────────────────────
# 3. Training Loop
# ─────────────────────────────────────────────────────────────────────────────

def count_params(params) -> int:
    return int(sum(x.size for x in jax.tree_util.tree_leaves(params)))


def make_seqs(imu_batch, T):
    """Expand (B, 6) mean IMU to (T, B, 6) time-major IMU sequence."""
    return jnp.broadcast_to(imu_batch[None], (T, *imu_batch.shape))


def train_model(name: str) -> dict:
    transformed, mode = _make_transformed(name)

    # ── Init ────────────────────────────────────────────────────────────────
    key0 = jax.random.PRNGKey(SEED)
    xb0, ub0, _ = make_batch_ab(Xtr_j, Utr_j, Ytr_z, BATCH_SIZE, key0)
    T0 = xb0.shape[0]

    if mode == "imu":
        params = transformed.init(key0, xb0, ub0)
        def _apply(p, xb, ub): return transformed.apply(p, xb, ub)
    elif mode == "recurrent":
        imu_seq0  = make_seqs(ub0, T0)
        traj_seq0 = jnp.zeros((T0, BATCH_SIZE, OUTPUT_DIM))
        params = transformed.init(key0, xb0, imu_seq0, traj_seq0)
        def _apply(p, xb, ub):
            T = xb.shape[0]; B = xb.shape[1]
            return transformed.apply(p, xb, make_seqs(ub, T),
                                     jnp.zeros((T, B, OUTPUT_DIM)))
    else:  # kalman
        imu_seq0 = make_seqs(ub0, T0)
        params = transformed.init(key0, xb0, imu_seq0)
        def _apply(p, xb, ub):
            return transformed.apply(p, xb, make_seqs(ub, xb.shape[0]))

    n_params  = count_params(params)
    opt       = optax.adamw(LR, weight_decay=1e-4)
    opt_state = opt.init(params)

    # ── JIT step ──────────────────────────────────────────────────────────────
    @jax.jit
    def train_step(params, opt_state, xb, ub, yb):
        def loss_fn(p):
            pred, aux = _apply(p, xb, ub)
            return jnp.mean((pred - yb) ** 2), (pred, aux)
        (loss, (pred, aux)), grads = jax.value_and_grad(
            loss_fn, has_aux=True)(params)
        updates, new_opt = opt.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_opt, loss, pred, aux

    @jax.jit
    def infer(params, xb, ub):
        return _apply(params, xb, ub)

    # ── Epoch loop ──────────────────────────────────────────────────────────
    history = defaultdict(list)
    key     = jax.random.PRNGKey(SEED + 77)

    for epoch in range(1, EPOCHS + 1):
        key, sk = jax.random.split(key)
        xb, ub, yb = make_batch_ab(Xtr_j, Utr_j, Ytr_z, BATCH_SIZE, sk)
        params, opt_state, tr_mse, _, tr_aux = train_step(
            params, opt_state, xb, ub, yb)

        # Validation
        va_maes = []
        for xvb, uvb, yvb in iter_batches(Xva_j, Uva_j, Yva_z):
            pv, _ = infer(params, xvb, uvb)
            va_maes.append(np.mean(np.abs(np.asarray(pv) - np.asarray(yvb)), axis=0))
        va_mae  = np.mean(va_maes, axis=0) if va_maes else np.zeros(OUTPUT_DIM)
        va_rmse = float(np.sqrt(np.mean(va_mae**2)))
        sr      = float(np.mean(np.asarray(
            tr_aux.get("spike_rate", jnp.zeros(1)))))

        history["epoch"].append(epoch)
        history["train_mse"].append(float(tr_mse))
        history["val_rmse"].append(va_rmse)
        history["val_mae_per_axis"].append(va_mae.tolist())
        history["spike_rate"].append(sr)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  [{name}] ep {epoch:3d} | train_mse={float(tr_mse):.4f} "
                  f"val_rmse={va_rmse:.4f} spike_rate={sr:.3f}")

    # ── Test evaluation ───────────────────────────────────────────────────────
    te_preds = []
    for xtb, utb, ytb in iter_batches(Xte_j, Ute_j, Yte_z):
        pt, _ = infer(params, xtb, utb)
        te_preds.append(np.asarray(pt))

    te_z      = np.concatenate(te_preds, axis=0)
    te_m      = te_z * Y_std + Y_mean                 # denorm → metres
    gt_m      = np.asarray(Yte_j[:te_z.shape[0]]) * Y_std + Y_mean
    te_mae    = np.mean(np.abs(te_m - gt_m), axis=0)
    te_rmse   = float(np.sqrt(np.mean(te_mae**2)))

    # ── Innovation norm (Kalman only) ─────────────────────────────────────────
    demo_xb, demo_ub, _ = make_batch_ab(Xte_j, Ute_j, Yte_z,
                                         min(BATCH_SIZE, Xte_j.shape[0]),
                                         jax.random.PRNGKey(99))
    _, demo_aux = infer(params, demo_xb, demo_ub)

    # ── Latency -─────────────────────────────────────────────────────────────
    infer(params, demo_xb, demo_ub)   # warmup
    t0 = time.perf_counter()
    for _ in range(50):
        jax.block_until_ready(infer(params, demo_xb, demo_ub))
    latency_ms = (time.perf_counter() - t0) / 50 * 1000

    return {
        "name":             name,
        "history":          dict(history),
        "final_mae_per_axis": te_mae.tolist(),
        "test_rmse":        te_rmse,
        "spike_rate":       history["spike_rate"][-1],
        "n_params":         n_params,
        "latency_ms":       latency_ms,
        "test_preds_m":     te_m,
        "test_gt_m":        gt_m,
        "demo_aux":         {k: float(np.mean(np.asarray(v))) for k, v in demo_aux.items()
                             if v is not None},
    }


# ─────────────────────────────────────────────────────────────────────────────
# 4. Run All Models
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for model_name in ["IMUConditioned", "RecurrentFusion", "KalmanFusion"]:
    print(f"\n{'='*60}\n  {model_name}\n{'='*60}")
    results[model_name] = train_model(model_name)

print("\n\n── Final Test RMSE ─────────────────────────────────────────────")
for nm, res in results.items():
    print(f"  {nm:<22}  RMSE={res['test_rmse']:.4f}  "
          f"params={res['n_params']:,}  latency={res['latency_ms']:.1f}ms  "
          f"spike_rate={res['spike_rate']:.3f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 1 – Learning Curves (train MSE, val RMSE, spike rate)
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for name, res in results.items():
    h = res["history"]
    c = MODEL_COLORS.get(name, "#333")
    ep = h["epoch"]
    axes[0].plot(ep, h["train_mse"],  label=name, color=c, lw=1.8)
    axes[1].plot(ep, h["val_rmse"],   label=name, color=c, lw=1.8)
    axes[2].plot(ep, h["spike_rate"], label=name, color=c, lw=1.8, ls="--")

for ax, title, yl in zip(axes,
        ["Train MSE (z-scored)", "Val RMSE (z-scored)", "Spike Rate"],
        ["MSE", "RMSE", "spikes/neuron/step"]):
    ax.set_title(title, fontweight="bold"); ax.set_xlabel("Epoch"); ax.set_ylabel(yl)
    ax.legend(fontsize=8)

fig.suptitle(f"Ablation Learning Curves — {RECORDING}", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "ablation_learning_curves.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 2 – Radar Chart: Multi-metric Model Comparison
# ─────────────────────────────────────────────────────────────────────────────
# Metrics: test_rmse (lower→better), spike_rate (lower→better),
#          latency_ms (lower→better), log10(n_params) (lower→better)
# We normalise each metric to [0,1] (1=best) for the radar.

metric_keys   = ["Test RMSE", "Spike Rate", "Latency (ms)", "log₁₀(params)"]
raw_vals = {
    name: [
        res["test_rmse"],
        res["spike_rate"],
        res["latency_ms"],
        np.log10(max(res["n_params"], 1)),
    ]
    for name, res in results.items()
}

# Normalise: lower raw → higher radar score
arr   = np.array(list(raw_vals.values()))
worst = arr.max(axis=0)
best  = arr.min(axis=0)
eps   = 1e-8
norm  = {name: 1.0 - (np.array(raw_vals[name]) - best) / (worst - best + eps)
         for name in results}

N_axes = len(metric_keys)
angles = np.linspace(0, 2 * np.pi, N_axes, endpoint=False).tolist()
angles += angles[:1]   # close polygon

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
for name, scores in norm.items():
    vals = scores.tolist() + scores[:1].tolist()
    ax.plot(angles, vals, "o-", lw=2, label=name, color=MODEL_COLORS.get(name, "#555"))
    ax.fill(angles, vals, alpha=0.1, color=MODEL_COLORS.get(name, "#555"))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_keys, fontsize=9)
ax.set_ylim(0, 1)
ax.set_title("Multi-Metric Comparison\n(outer = better)", fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "ablation_radar_chart.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 3 – Predicted vs Ground-Truth Translation (test set)
# ─────────────────────────────────────────────────────────────────────────────
axis_names = ["tx (m)", "ty (m)", "tz (m)"]
n_models   = len(results)

fig, axes = plt.subplots(n_models, 3, figsize=(13, 3.5 * n_models), squeeze=False)

for row, (name, res) in enumerate(results.items()):
    gt   = res["test_gt_m"]
    pred = res["test_preds_m"]
    n    = min(gt.shape[0], pred.shape[0])
    c    = MODEL_COLORS.get(name, "#e41a1c")
    for col in range(3):
        ax = axes[row][col]
        ax.plot(gt[:n, col],   color="#555555", lw=1,   label="GT",   alpha=0.8)
        ax.plot(pred[:n, col], color=c,         lw=1.2, label=name)
        if row == 0:
            ax.set_title(axis_names[col], fontweight="bold")
        if col == 0:
            ax.set_ylabel(name, fontsize=8)
        if row == n_models - 1:
            ax.set_xlabel("sample index")
        if row == 0 and col == 0:
            ax.legend(fontsize=7)

fig.suptitle(f"Test-Set Trajectory Predictions — {RECORDING}", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "ablation_trajectory.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 4 – Complexity vs Accuracy + Auxiliary Signal Analysis
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: param count vs test RMSE
for name, res in results.items():
    axes[0].scatter(np.log10(max(res["n_params"], 1)), res["test_rmse"],
                    s=200, label=name, color=MODEL_COLORS.get(name, "#888"),
                    edgecolors="white", linewidths=2, zorder=5)
    axes[0].annotate(name,
                     (np.log10(max(res["n_params"], 1)), res["test_rmse"]),
                     textcoords="offset points", xytext=(6, 4), fontsize=8)
axes[0].set_xlabel("log₁₀(Parameter Count)")
axes[0].set_ylabel("Test RMSE (metres)")
axes[0].set_title("Accuracy vs Model Capacity", fontweight="bold")
axes[0].legend(fontsize=8)

# Panel B: Auxiliary signal introspection
# Shows innovation_norm (Kalman), spike_rate (all), etc.
aux_keys = ["spike_rate", "innovation_norm"]  # not all models have all keys
bar_x    = np.arange(len(results))
width    = 0.35

ax2     = axes[1]
colors  = [MODEL_COLORS.get(n, "#888") for n in results]
rmse_v  = [res["test_rmse"] for res in results.values()]
spike_v = [res["spike_rate"] for res in results.values()]

bars1 = ax2.bar(bar_x - width/2, rmse_v,  width, label="Test RMSE (m)",
                color=[c + "aa" for c in colors], edgecolor="white")
ax2b  = ax2.twinx()
bars2 = ax2b.bar(bar_x + width/2, spike_v, width, label="Spike Rate",
                 color=colors, edgecolor="white", alpha=0.8)

ax2.set_xticks(bar_x)
ax2.set_xticklabels(list(results.keys()), rotation=15, ha="right", fontsize=8)
ax2.set_ylabel("Test RMSE (metres)", color="#333")
ax2b.set_ylabel("Mean Spike Rate", color="#555")
ax2.set_title("Accuracy vs Energy Proxy", fontweight="bold")
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper right")

fig.suptitle("Model Complexity & Auxiliary Analysis", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "ablation_complexity_analysis.png", bbox_inches="tight")
plt.show()

# ── Print aux dict (innovation norm for Kalman etc.) ─────────────────────────
print("\n── Auxiliary Signals ───────────────────────────────────────────────────")
for name, res in results.items():
    print(f"  {name:<22}  {res['demo_aux']}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Summary Table + Persist Results
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for name, res in results.items():
    rows.append({
        "model":       name,
        "tx_mae (m)":  round(res["final_mae_per_axis"][0], 4),
        "ty_mae (m)":  round(res["final_mae_per_axis"][1], 4),
        "tz_mae (m)":  round(res["final_mae_per_axis"][2], 4),
        "test_rmse":   round(res["test_rmse"], 4),
        "spike_rate":  round(res["spike_rate"], 4),
        "n_params":    res["n_params"],
        "latency_ms":  round(res["latency_ms"], 2),
    })

df = pd.DataFrame(rows).set_index("model").sort_values("test_rmse")
print("\n── Ablation Benchmark Table ────────────────────────────────────────────")
print(df.to_string())

# ── Persist ──────────────────────────────────────────────────────────────────
run_record = {
    "timestamp":  datetime.datetime.utcnow().isoformat() + "Z",
    "experiment": "tumvie_new_models_ablation",
    "recording":  RECORDING,
    "epochs":     EPOCHS,
    "n_samples":  N_SAMPLES,
    "results":    {name: {k: v for k, v in res.items()
                   if k not in ("test_preds_m", "test_gt_m", "demo_aux")}
                   for name, res in results.items()},
}
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_FILE, "a") as f:
    f.write(json.dumps(run_record) + "\n")
print(f"\n✓ Results appended to {RESULTS_FILE}")